# Fraud Detection Experiments - Neo4j Graph Features on Vertex AI

## Before you run

**Step 1 - Data is already in GCS** at `gs://neo4j-fraud-detection/artifacts/demo_dataset.parquet`

**Step 2 - Run all cells:** Runtime > Run all

---

### What the experiments compare

| | Experiment 1 | Experiment 2 |
|---|---|---|
| **Name** | Tabular Baseline | Graph-Enhanced |
| **Features** | Raw transaction columns only | Tabular + graph features + FastRP embeddings |
| **Graph context** | None | Entity fraud rates, temporal card chain, 64-dim node embeddings |

Both experiments use the same LightGBM architecture and hyperparameters. The only difference is the feature set.

In [ ]:
!pip install lightgbm --quiet

In [ ]:
# ── Data path ──────────────────────────────────────────────────────────────
# Vertex AI / Colab Enterprise (GCS):
DATASET_PATH = 'gs://neo4j-fraud-detection/artifacts/demo_dataset.parquet'
# Local testing: '../artifacts/demo_dataset.parquet'

RESULTS_GCS = None  # optional: 'gs://neo4j-fraud-detection/results/'

LGBM_PARAMS = dict(
    n_estimators  = 500,
    learning_rate = 0.05,
    num_leaves    = 63,
    random_state  = 42,
    n_jobs        = -1,
    verbose       = -1,
)
EARLY_STOP = 50
SPLIT_DT   = 12_528_000

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import json
import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, ConfusionMatrixDisplay,
)
from sklearn.decomposition import PCA
import lightgbm as lgb
from matplotlib.patches import Patch
print('Imports OK.')

---
## 1. Load Dataset

`demo_dataset.parquet` is a single merged file (274 MB) containing:
- 590,540 transaction rows
- Raw tabular features (394 cols from the IEEE-CIS dataset)
- 22 graph features extracted from Neo4j (entity fraud rates, proxy flag, temporal card chain)
- 64 FastRP node embedding dimensions computed by Neo4j GDS

In [ ]:
print(f'Loading: {DATASET_PATH}')
df = pd.read_parquet(DATASET_PATH)
print(f'Shape  : {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Fraud  : {df["isFraud"].sum():,} ({df["isFraud"].mean():.2%} of transactions)')
df[['TransactionID','TransactionAmt','isFraud','card_fraud_rate','prev_card_is_fraud','emb_0','emb_1']].head(4)

---
## 2. Feature Engineering and Train/Val Split

In [ ]:
# Label-encode categorical columns for LightGBM
CAT_COLS = ['card4', 'card6', 'ProductCD', 'P_emaildomain', 'R_emaildomain']
for col in CAT_COLS:
    if col in df.columns:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str).fillna('nan'))

GRAPH_COLS = [
    'card_tx_count',        'card_fraud_count',        'card_fraud_rate',
    'payer_email_tx_count', 'payer_email_fraud_count', 'payer_email_fraud_rate',
    'recip_email_tx_count', 'recip_email_fraud_count', 'recip_email_fraud_rate',
    'billing_tx_count',     'billing_fraud_count',     'billing_fraud_rate',
    'device_tx_count',      'device_fraud_count',      'device_fraud_rate',
    'os_browser_tx_count',  'os_browser_fraud_count',  'os_browser_fraud_rate',
    'is_proxy', 'proxy_fraud_rate', 'prev_card_is_fraud', 'prev_card_dt_gap',
]
GRAPH_COLS = [c for c in GRAPH_COLS if c in df.columns]
EMB_COLS   = [f'emb_{i}' for i in range(64) if f'emb_{i}' in df.columns]
EXCLUDE    = {'TransactionID', 'TransactionDT', 'isFraud'}
TAB_COLS   = [c for c in df.columns if c not in EXCLUDE and c not in GRAPH_COLS and c not in EMB_COLS]
ALL_COLS   = TAB_COLS + GRAPH_COLS + EMB_COLS

print(f'Tabular features  : {len(TAB_COLS)}')
print(f'Graph features    : {len(GRAPH_COLS)}')
print(f'Embedding dims    : {len(EMB_COLS)}')
print(f'Total (enhanced)  : {len(ALL_COLS)}')

train = df[df['TransactionDT'] <= SPLIT_DT].copy()
val   = df[df['TransactionDT'] >  SPLIT_DT].copy()
SCALE_POS = (train['isFraud'] == 0).sum() / (train['isFraud'] == 1).sum()

print(f'\nTrain : {len(train):>7,} rows  |  fraud: {train["isFraud"].mean():.2%}')
print(f'Val   : {len(val):>7,} rows  |  fraud: {val["isFraud"].mean():.2%}')
print(f'Class imbalance weight: {SCALE_POS:.1f}')

---
## Experiment 1 - Tabular Baseline

LightGBM trained on raw tabular features only. Each transaction is scored in isolation - no knowledge of what other transactions share the same card, device, or email domain.

In [ ]:
print('Training tabular baseline ...')
print(f'  Features : {len(TAB_COLS)} tabular columns')

X_tr  = train[TAB_COLS];  y_tr  = train['isFraud']
X_val = val[TAB_COLS];    y_val = val['isFraud']

clf_base = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'scale_pos_weight': SCALE_POS})
clf_base.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(50)],
)
print(f'Done. Best iteration: {clf_base.best_iteration_}')

In [ ]:
prob_base = clf_base.predict_proba(X_val)[:, 1]
roc_base  = roc_auc_score(y_val, prob_base)
pr_base   = average_precision_score(y_val, prob_base)

best_f1, best_t = 0, 0.5
for t in np.arange(0.05, 0.95, 0.01):
    s = f1_score(y_val, prob_base >= t, zero_division=0)
    if s > best_f1: best_f1, best_t = s, t
thresh_base = best_t
pred_base   = (prob_base >= thresh_base).astype(int)
f1_base     = f1_score(y_val, pred_base)
prec_base   = precision_score(y_val, pred_base, zero_division=0)
rec_base    = recall_score(y_val, pred_base)

print('-- Experiment 1: Tabular Baseline --')
print(f'ROC-AUC   : {roc_base:.4f}')
print(f'PR-AUC    : {pr_base:.4f}  <-- primary metric')
print(f'F1        : {f1_base:.4f}   (threshold={thresh_base:.2f})')
print(f'Precision : {prec_base:.4f}')
print(f'Recall    : {rec_base:.4f}')

---
## Experiment 2 - Graph-Enhanced Model

Same LightGBM architecture. Feature set extended with:
- **22 graph features** from Neo4j: entity-level fraud rates per card, device, billing address, email domain, OS/browser; proxy flag; `prev_card_is_fraud` temporal chain signal
- **64 FastRP embedding dimensions** from Neo4j GDS: each transaction's structural fingerprint in the fraud network

In [ ]:
print('Training graph-enhanced model ...')
print(f'  Features : {len(TAB_COLS)} tabular + {len(GRAPH_COLS)} graph + {len(EMB_COLS)} embeddings = {len(ALL_COLS)} total')

X_tr_g  = train[ALL_COLS].fillna(-1);  y_tr_g  = train['isFraud']
X_val_g = val[ALL_COLS].fillna(-1);    y_val_g = val['isFraud']

clf_graph = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'scale_pos_weight': SCALE_POS})
clf_graph.fit(
    X_tr_g, y_tr_g,
    eval_set=[(X_val_g, y_val_g)],
    callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False), lgb.log_evaluation(50)],
)
print(f'Done. Best iteration: {clf_graph.best_iteration_}')

In [ ]:
prob_graph = clf_graph.predict_proba(X_val_g)[:, 1]
roc_graph  = roc_auc_score(y_val_g, prob_graph)
pr_graph   = average_precision_score(y_val_g, prob_graph)

best_f1, best_t = 0, 0.5
for t in np.arange(0.05, 0.95, 0.01):
    s = f1_score(y_val_g, prob_graph >= t, zero_division=0)
    if s > best_f1: best_f1, best_t = s, t
thresh_graph = best_t
pred_graph   = (prob_graph >= thresh_graph).astype(int)
f1_graph     = f1_score(y_val_g, pred_graph)
prec_graph   = precision_score(y_val_g, pred_graph, zero_division=0)
rec_graph    = recall_score(y_val_g, pred_graph)

print('-- Experiment 2: Graph-Enhanced --')
print(f'ROC-AUC   : {roc_graph:.4f}')
print(f'PR-AUC    : {pr_graph:.4f}  <-- primary metric')
print(f'F1        : {f1_graph:.4f}   (threshold={thresh_graph:.2f})')
print(f'Precision : {prec_graph:.4f}')
print(f'Recall    : {rec_graph:.4f}')

---
## Results

In [ ]:
metrics      = ['ROC-AUC', 'PR-AUC', 'F1', 'Precision', 'Recall']
base_scores  = [roc_base,  pr_base,  f1_base,  prec_base,  rec_base]
graph_scores = [roc_graph, pr_graph, f1_graph, prec_graph, rec_graph]

results = pd.DataFrame({
    'Metric':           metrics,
    'Tabular Baseline': base_scores,
    'Graph-Enhanced':   graph_scores,
})
results['Delta'] = results['Graph-Enhanced'] - results['Tabular Baseline']
results['Lift']  = (results['Delta'] / results['Tabular Baseline'] * 100).map('{:+.1f}%'.format)
print(results.round(4).to_string(index=False))

fraud_base  = int(pred_base[y_val == 1].sum())
fraud_graph = int(pred_graph[y_val_g == 1].sum())
total_fraud = int(y_val.sum())
print(f'\nFraud caught - Baseline       : {fraud_base:,}  ({fraud_base/total_fraud:.1%})')
print(f'Fraud caught - Graph-Enhanced : {fraud_graph:,}  ({fraud_graph/total_fraud:.1%})')
print(f'Extra fraud caught            : +{fraud_graph - fraud_base}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Fraud Detection Experiments: Tabular Baseline vs Graph-Enhanced', fontsize=13)

x = np.arange(len(metrics)); w = 0.35
b1 = axes[0].bar(x - w/2, base_scores,  w, label='Tabular Baseline', color='#3498db')
b2 = axes[0].bar(x + w/2, graph_scores, w, label='Graph-Enhanced',   color='#e74c3c')
axes[0].set_xticks(x); axes[0].set_xticklabels(metrics, rotation=20, ha='right')
axes[0].set_ylim(0.5, 1.05); axes[0].legend(fontsize=9); axes[0].set_title('All Metrics')
axes[0].bar_label(b1, fmt='%.3f', fontsize=8, padding=2)
axes[0].bar_label(b2, fmt='%.3f', fontsize=8, padding=2)

axes[1].bar(['Tabular\nBaseline','Graph-\nEnhanced'], [pr_base, pr_graph],
            color=['#3498db','#e74c3c'], width=0.4)
axes[1].set_ylim(0, 1.0); axes[1].set_title('PR-AUC (primary metric)')
for i, v in enumerate([pr_base, pr_graph]):
    axes[1].text(i, v+0.012, f'{v:.4f}', ha='center', fontsize=13, fontweight='bold')
axes[1].set_xlabel(f'Graph improvement: {(pr_graph-pr_base)/pr_base*100:+.1f}% PR-AUC', fontsize=11)

feat_imp = (pd.DataFrame({'feature': clf_graph.feature_name_, 'importance': clf_graph.feature_importances_})
            .sort_values('importance', ascending=False).head(20))
color_map = {'Graph': '#e74c3c', 'Embedding': '#f39c12', 'Tabular': '#3498db'}
def ftype(n):
    return 'Graph' if n in GRAPH_COLS else ('Embedding' if n in EMB_COLS else 'Tabular')
axes[2].barh(feat_imp['feature'], feat_imp['importance'],
             color=feat_imp['feature'].map(ftype).map(color_map).tolist())
axes[2].invert_yaxis(); axes[2].set_title('Top 20 Feature Importances')
axes[2].legend(handles=[Patch(facecolor=c, label=t) for t,c in color_map.items()],
               loc='lower right', fontsize=9)
plt.tight_layout(); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(confusion_matrix(y_val,   pred_base),  display_labels=['Legit','Fraud']).plot(ax=axes[0], colorbar=False, cmap='Blues')
ConfusionMatrixDisplay(confusion_matrix(y_val_g, pred_graph), display_labels=['Legit','Fraud']).plot(ax=axes[1], colorbar=False, cmap='Reds')
axes[0].set_title(f'Experiment 1 - Tabular Baseline\nPR-AUC = {pr_base:.4f}')
axes[1].set_title(f'Experiment 2 - Graph-Enhanced\nPR-AUC = {pr_graph:.4f}')
plt.tight_layout(); plt.show()

In [ ]:
# FastRP embedding PCA visualization
sample = train.sample(n=min(10_000, len(train)), random_state=42)
X_2d = PCA(n_components=2, random_state=42).fit_transform(sample[EMB_COLS].fillna(0).values)
y_s  = sample['isFraud'].values

fig, ax = plt.subplots(figsize=(10, 6))
for lbl, name, col, a, s in [(0,'Legitimate','#3498db',0.2,4),(1,'Fraud','#e74c3c',0.7,8)]:
    m = y_s == lbl
    ax.scatter(X_2d[m,0], X_2d[m,1], c=col, alpha=a, s=s, label=f'{name} ({m.sum():,})')
ax.set_title('FastRP Node Embeddings - PCA Projection (64D -> 2D, 10K sample)', fontsize=12)
ax.set_xlabel('PC1'); ax.set_ylabel('PC2'); ax.legend(fontsize=11, markerscale=3)
plt.tight_layout(); plt.show()